# Training an SHC soft sensor

**Goal:** train a model that predicts a plant's specific heat consumption
(SHC) from its sensor readings.

SHC is the energy used to make one unit of clinker, in kcal/kg. Lower is
better -- less fuel burnt means less CO2.

A *soft sensor* is a model that predicts something you can't easily measure,
using things you can. Here the inputs are sensor readings, and the output is
SHC.

Work through the sections below and fill in each `TODO`.
Run a cell with **Shift+Enter**.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

## 1. Load the data

Load the plant data from ClickHouse using the loader in
`src/python/data/clickhouse.py`. Pick a sensible date range and only ask for
the features you need -- the tables are big, and loading everything is slow.

In [ ]:
# TODO: load the plant data.
#
# from src.python.data.clickhouse import ClickHouseConfig, PlantDataLoader
# import datetime as dt
#
# loader = PlantDataLoader(ClickHouseConfig.from_environment())
# data = loader.load(
#     table=...,
#     features=[...],
#     start=dt.datetime(...),
#     end=dt.datetime(...),
# )

## 2. Look at your data

Always look before you model. You are checking for:

- How many rows did you get?
- Are there gaps or missing values?
- Does anything look obviously wrong (negative values, huge spikes)?
- Was the plant actually running? A plant that is off produces junk data.

In [ ]:
# TODO: inspect the data.
# data.head()
# data.describe()
# data.isna().sum()

## 3. Choose your target and features

The **target** (`y`) is SHC -- the thing you want to predict.

The **features** (`X`) are the inputs the model learns from.

One important rule: **leave out the fuel we control.**

Gigaton's models recommend changes to fuel. If the model could see the fuel
actually used, predicting SHC would be trivial and useless -- it would just
tell us what happened. By hiding it, the model learns what SHC we *should*
expect given everything else, and we can compare that against what the plant
really achieved. That difference is our impact.

In [ ]:
# TODO: split into features and target.
#
# target = "..."
# features = [...]   # everything except the fuel we control
#
# X = data[features]
# y = data[target]

## 4. Split into training and test sets

The model must be scored on new data it has never seen!

Use `shuffle=False`. This is time series data, and shuffling would let the
model learn from the future to predict the past.

In [ ]:
# TODO: split the data.
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, shuffle=False
# )

## 5. Train a model

Start with `LinearRegression` -- it is simple and easy to reason about.
Once it works, try `RandomForestRegressor` and see whether it does better.

Always get a simple model working first. It gives you a baseline to beat.

In [ ]:
# TODO: train the model.
#
# model = LinearRegression()
# model.fit(X_train, y_train)

## 6. How good is it?

Two useful numbers:

- **Mean absolute error** -- how far off you are on average, in kcal/kg.
  Smaller is better.
- **R squared** -- how much of the variation the model explains, from 0 to 1.
  Bigger is better.

Ask yourself: is this error small enough to be useful to a plant?

In [ ]:
# TODO: score the model on the test set.
#
# predictions = model.predict(X_test)
# print(f"Mean absolute error: {mean_absolute_error(y_test, predictions):.1f} kcal/kg")
# print(f"R squared: {r2_score(y_test, predictions):.2f}")

## 7. Plot predicted vs actual

Numbers alone hide problems. Two plots worth making:

1. Predicted and actual SHC over time, on the same axes.
2. Predicted against actual as a scatter -- points on the diagonal are
   perfect predictions.

In [ ]:
# TODO: plot the predictions against the real values.

## 8. Which features mattered?

Have a look at what the model leaned on. For `LinearRegression` that is
`model.coef_`; for `RandomForestRegressor` it is
`model.feature_importances_`.

If something surprising is at the top, that is worth understanding -- it can
reveal a mistake, or teach you something real about the plant.

In [ ]:
# TODO: inspect feature importance.

## Next steps

Once this works, move the good bits into `src/python/soft_sensor/shc.py` so
they can be tested, then move on to Phase 3 and get your predictions into a
dashboard.

Notebooks are for exploring; `src/` is for code that needs to keep working.